In [1]:
# 1. 导入必要的库
from vnpy.trader.setting import SETTINGS
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import  HistoryRequest
from vnpy.trader.datafeed import get_datafeed
from vnpy.alpha.lab import AlphaLab
from vnpy.trader.database import DB_TZ
from vnpy.alpha import  logger
from datetime import datetime, timedelta
import rqdatac as rq
from pathlib import Path
from tqdm import tqdm
import json

In [2]:
# 2. 配置RQData数据服务

# 检查是否已配置用户名密码
print('当前数据服务配置:')
print('datafeed.username:', SETTINGS.get('datafeed.username', '未设置'))
print('datafeed.password:', SETTINGS.get('datafeed.password', '未设置'))

# 初始化数据服务
datafeed = get_datafeed()
print(f'数据服务类型: {datafeed.__class__.__name__}')

# 尝试初始化
inited = datafeed.init(output=print)
print(f'初始化结果: {inited}')

当前数据服务配置:
datafeed.username: license
datafeed.password: cilEC1HhQzr7r54PMYYmzzdKoIC6tItQ2c2TpCZPJOP22R7FftDt8evpY4pcb0_OiFTcngfpd9zs0LKslCQqoISMXbqG9dLk7L5999PgeVgi6iBD8ng2LoPGWgHjsYQb3jbvtYNLUtoukRyapl41y7sSFvI6vFnOrNoPnbwe-pg=T7HC7VpqgXpjVes9jwMsDJMxRLM_zhVDxCaSAhWcqYVSoI4HGD4xLpfp4hwVVZDZqlCQWmx9vHpdRBTI77QbNkdAIQVZdos1EgXK78APi6TjdTG3Zk_UiVgz_0MUhMq3rBSAXfLuolbc9mpRV95AK8afgK7jTzZaP--T9soxgfE=
数据服务类型: RqdataDatafeed
RQData数据服务初始化失败：this license is only allowed to access through the education network
初始化结果: False


In [3]:
# 3. 路径配置

BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'
MINUTE_PATH = LAB_PATH / 'minute'
DAILY_PATH = LAB_PATH / 'daily'

# 获取MF_Lab
lab = AlphaLab(str(LAB_PATH))

In [4]:
# 4. 数据下载

vt_index_symbol = "000300.SSE"
rq_index_symbol = "000300.XSHG"

# 总时间跨度
start = datetime(2018, 1, 1)
end = datetime(2026,5,8)
extended_days = 370
extended_years = 8
extended_start = start - timedelta(years=extended_days)
print(extended_start)

interval1 = Interval.MINUTE      #数据频率

# 之前的跨度
start2 = datetime(2018, 1, 1)
end2 = datetime(2026, 4, 23)

# 回测跨度 回测需要日线数据算收益
test_start = datetime(2025, 1, 1)
test_end = end

test_start2 = datetime(2025, 1, 1)
test_end2 = end2
interval2 = Interval.DAILY


trading_days = rq.get_trading_dates(  # 获取交易日
            start,
            end,
            market="cn"  # 'cn' 代表中国证券市场
        )
trading_days_str = [d.isoformat() for d in trading_days]
with open(lab.trading_days_path, "w+") as f:
    json.dump(trading_days_str, f, indent=2)


2026-05-10 20:43:37.124265


In [5]:
# 4.1 下载成分股列表

# 下载总时间跨度的动态沪深300股票池股票代码
data = rq.index_components(rq_index_symbol, start_date=datetime(2010,1,1), end_date=end)

# 将rq的合约代码转化内vnpy的合约代码
vt_index_components = {}
for dt, rq_symbols in data.items():
    vt_symbols: list = []

    for rq_symbol in rq_symbols:
        vt_symbol = rq_symbol.replace("XSHG", "SSE").replace("XSHE", "SZSE")
        vt_symbols.append(vt_symbol)

    vt_index_components[dt.strftime("%Y-%m-%d")] = vt_symbols    #index_components = {"%Y-%m-%d":[成分股列表]，....}

# 保存
lab.save_component_data(vt_index_symbol, vt_index_components)

In [6]:
# 加载成分股代码
component_symbols1 = lab.load_component_symbols(vt_index_symbol, start, end)
component_symbols2 = lab.load_component_symbols(vt_index_symbol, start2, end2)
print(len(component_symbols1))
print(len(component_symbols2))

553
553


In [5]:
component_symbols = lab.load_component_symbols(vt_index_symbol, start, end)

In [6]:
task_symbols = component_symbols

In [6]:
for vt_symbol in component_symbols:
    lab.add_contract_setting(
        vt_symbol,
        long_rate=5/10000,
        short_rate=15/10000,
        size=1,
        pricetick=0.0001,
    )

In [7]:
start = start.replace(tzinfo=DB_TZ)    # 下载要标明时区
end = end.replace(tzinfo=DB_TZ)
extend_start = extend_start.replace(tzinfo=DB_TZ)
start2 = start2.replace(tzinfo=DB_TZ)    # 下载要标明时区
end2 = end2.replace(tzinfo=DB_TZ)

In [ ]:
#最新的股票
task_symbols = [s for s in component_symbols1 if s not in component_symbols2]    # 目标股票
print(len(task_symbols))

In [ ]:
#--------------------下载新的股票的全时间跨度的k线(mink)

# 获取所有 .parquet 文件的名称部分（不含扩展名）
# done_symbols = [f.stem for f in Path(MINUTE_PATH).glob("*.parquet")]
# task_symbols = [vt_symbol for vt_symbol in task_symbols if vt_symbol not in done_symbols]

n = 0

for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval1)
    bars = datafeed.query_bar_history(req)
    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

In [11]:
#--------------------下载新的股票的全时间跨度的k线(日k)
n = 0
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), datetime(2013,1,1).replace(tzinfo=DB_TZ), datetime(2017,1,1).replace(tzinfo=DB_TZ), interval2)
    bars = datafeed.query_bar_history(req)
    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

  0%|          | 0/553 [00:00<?, ?it/s]

下载 601212.SSE ...
2026-05-10 20:48:13 下载601212.SSE数据失败


  0%|          | 1/553 [00:00<00:57,  9.64it/s]

下载 601077.SSE ...
2026-05-10 20:48:13 下载601077.SSE数据失败
下载 002938.SZSE ...
2026-05-10 20:48:13 下载002938.SZSE数据失败


  1%|          | 3/553 [00:00<00:40, 13.51it/s]

...

股票428
  ->  971 条K线
下载 601608.SSE ...
股票429
  ->  971 条K线
下载 600837.SSE ...


100%|█████████▉| 552/553 [01:09<00:00,  7.07it/s]

股票430
  ->  971 条K线
下载 600763.SSE ...
股票431
  ->  971 条K线
下载 002202.SZSE ...


100%|██████████| 553/553 [01:09<00:00,  7.96it/s]

股票432
  ->  971 条K线
剩余121只未下载


In [ ]:
#--------------------下载新的股票的全时间跨度的k线(日k)
n = 0
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval2)
    bars = datafeed.query_bar_history(req)
    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

In [8]:
#--------------------旧的股票只缺end2到end的k线
task_symbols = component_symbols2

In [9]:
print(len(task_symbols))

553


In [9]:
##--------------------下载旧的股票的剩下时间跨度的k线(mink)
n = 0
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), end2, end, interval1)
    bars = datafeed.query_bar_history(req)

    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

  0%|          | 0/553 [00:00<?, ?it/s]

下载 601006.SSE ...


  0%|          | 2/553 [00:01<06:59,  1.31it/s]

股票1
  ->  1440 条K线
下载 002466.SZSE ...
股票2
  ->  1440 条K线


  1%|          | 3/553 [00:01<04:42,  1.95it/s]

...

股票536
  ->  1440 条K线
下载 600352.SSE ...


100%|█████████▉| 552/553 [02:03<00:00,  4.34it/s]

股票537
  ->  1440 条K线
下载 002050.SZSE ...


100%|██████████| 553/553 [02:04<00:00,  4.45it/s]

股票538
  ->  1440 条K线
剩余15只未下载


In [10]:
##--------------------下载旧的股票的剩下时间跨度的k线(日k)
task_symbols = task_symbols + [vt_index_symbol]
n = 0
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), end2, end , interval2)
    bars = datafeed.query_bar_history(req)

    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

  0%|          | 1/554 [00:00<01:02,  8.81it/s]

下载 601006.SSE ...
股票1
  ->  6 条K线
下载 002466.SZSE ...
股票2
  ->  6 条K线
下载 603806.SSE ...


  1%|          | 5/554 [00:00<00:42, 13.03it/s]

股票3
  ->  6 条K线
下载 600518.SSE ...
股票4
  ->  6 条K线
下载 000559.SZSE ...
股票5
  ->  6 条K线
下载 000166.SZSE ...
股票6
  ->  6 条K线
下载 600346.SSE ...


  2%|▏         | 9/554 [00:00<00:38, 13.98it/s]

...

股票530
  ->  6 条K线
下载 600118.SSE ...
股票531
  ->  6 条K线
下载 300782.SZSE ...
股票532
  ->  6 条K线
下载 000001.SZSE ...
股票533
  ->  6 条K线
下载 000596.SZSE ...


 99%|█████████▉| 551/554 [00:39<00:00, 15.69it/s]

股票534
  ->  6 条K线
下载 600905.SSE ...
股票535
  ->  6 条K线
下载 600153.SSE ...
股票536
  ->  6 条K线
下载 600352.SSE ...
股票537
  ->  6 条K线
下载 002050.SZSE ...


100%|██████████| 554/554 [00:39<00:00, 14.00it/s]

股票538
  ->  6 条K线
下载 000300.SSE ...
股票539
  ->  6 条K线
剩余15只未下载


In [9]:
quota = rq.user.get_quota()
print(quota)

{'bytes_used': 12748292, 'bytes_limit': 209715200.0, 'remaining_days': 477, 'license_type': 'EDU'}
